# Finding Signals — Dataset 2

Identify columns in **X** that provide signal (are correlated) to **y**, with estimates of signal size and p-values.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import os
import warnings
warnings.filterwarnings("ignore")

# Path to dataset2 — set this to where your dataset2 folder lives (e.g. "dataset2" or "/dataset2")
DATA_DIR = "dataset2"
if not os.path.isdir(DATA_DIR):
    DATA_DIR = "/dataset2"  # try absolute path
if not os.path.isdir(DATA_DIR):
    DATA_DIR = os.path.join(os.getcwd(), "dataset2")
print("Using DATA_DIR:", os.path.abspath(DATA_DIR))

## Load and merge data

In [ ]:
y_df = pd.read_csv(f"{DATA_DIR}/y.csv")
X_df = pd.read_csv(f"{DATA_DIR}/X.csv")

print("y.csv shape:", y_df.shape)
print("X.csv shape:", X_df.shape)

# Merge on id
df = X_df.merge(y_df, on="id", how="inner")
predictors = [c for c in X_df.columns if c.startswith("V")]
y_vec = df["y"].values
n = len(y_vec)
print(f"\nMerged rows: {len(df)}, Predictors: {predictors}")

## Identify columns that provide signal (correlated with y)

For each predictor we compute Pearson correlation with y and its p-value. We treat predictors as "signal" if the correlation is statistically significant (e.g. p < 0.05) and/or |r| is meaningfully large.

In [ ]:
results = []
for col in predictors:
    x_vec = df[col].astype(float).values
    # Pearson correlation and p-value (two-tailed)
    r, p = stats.pearsonr(x_vec, y_vec)
    # Signal size: standardized effect (correlation) and raw slope from simple regression
    slope, intercept, r_val, p_val, se = stats.linregress(x_vec, y_vec)
    # Standardized slope (beta) ≈ correlation when both are standardized
    x_std = (x_vec - x_vec.mean()) / (x_vec.std() or 1e-10)
    y_std = (y_vec - y_vec.mean()) / (y_vec.std() or 1e-10)
    beta_std, _, _, _, _ = stats.linregress(x_std, y_std)
    results.append({
        "predictor": col,
        "correlation_r": r,
        "p_value": p,
        "signal_size_slope": slope,
        "signal_size_std_beta": beta_std,
        "significant_005": p < 0.05,
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("p_value")
results_df

## Selected columns (those that provide signal)

We select predictors that are significantly correlated with y at α = 0.05.

In [ ]:
selected = results_df[results_df["significant_005"]]["predictor"].tolist()
print("Columns that provide signal (p < 0.05):", selected if selected else "None at α=0.05")

# If none significant, show top by |correlation|
if not selected:
    top = results_df.nsmallest(3, "p_value")
    print("\nTop 3 predictors by p-value (for reference):")
    print(top[["predictor", "correlation_r", "p_value"]].to_string(index=False))

## Signal size estimates and p-values (summary)

| Predictor | Correlation (r) | Signal size (std beta) | Signal size (slope) | p-value |

In [ ]:
summary = results_df[["predictor", "correlation_r", "signal_size_std_beta", "signal_size_slope", "p_value"]].copy()
summary.columns = ["Predictor", "Correlation (r)", "Signal size (std β)", "Signal size (slope)", "p-value"]
summary["p-value"] = summary["p-value"].map(lambda x: f"{x:.4f}" if x >= 0.0001 else f"{x:.2e}")
print(summary.to_string(index=False))

In [ ]:
# Final answer: selected predictors with signal size and p-value
print("=== Finding Signals — Dataset 2 ===\n")
print("Columns that provide signal:", selected)
print("\nSignal size (standardized β) and p-value per selected predictor:")
for _, row in results_df[results_df["significant_005"]].iterrows():
    print(f"  {row['predictor']}: β_std = {row['signal_size_std_beta']:.4f}, p = {row['p_value']:.4f}")
if not selected:
    print("  (No predictor had p < 0.05; see full table above for top candidates.)")